# Research Question 5: User Adoption Patterns

## 🎯 Research Question  
**"Who adopts coding agents (newcomers vs experienced developers)?"**

## 📋 Methodology
- **User Classification**: Identify newcomers vs experienced developers
- **Adoption Metrics**:
  - First-time vs repeat AI agent usage
  - User activity patterns over time
  - Agent preference by user experience level
  - Temporal adoption trends

## 🔍 Expected Insights
- Understand user demographics for AI agent adoption
- Identify patterns in how different user types engage with AI tools
- Track adoption trends and user retention
- Establish user experience correlations with AI usage

## 📊 Classification Strategy
- **Newcomers**: Users with recent account creation or low activity
- **Experienced**: Users with extensive contribution history
- **Metrics**: PR count, account age (when available), activity patterns

## 🎯 Success Metrics
- User classification accuracy
- Adoption rate by user type
- Temporal trends in user adoption
- Agent preference patterns by experience level

In [ ]:
# Setup for User Adoption Analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
from datetime import datetime, timedelta
from collections import Counter

# Add src directory to path and force reload for updated random sampling
sys.path.append('../src')
import importlib
if 'data_loader' in sys.modules:
    importlib.reload(sys.modules['data_loader'])
from data_loader import load_aidev

print("RQ5: User Adoption Patterns Analysis")
print("=" * 50)
print(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Load representative sample with random sampling to ensure all agents
print("\nLoading representative data sample...")
df = load_aidev(sample_size=50000)  # Random sample for all 5 agents
print(f"Loaded {len(df):,} PRs for user adoption analysis")

# Check agent representation
print(f"\nAgent representation in sample:")
agent_counts = df['agent'].value_counts()
for agent, count in agent_counts.items():
    percentage = (count / len(df)) * 100
    print(f"  {agent}: {count:,} PRs ({percentage:.1f}%)")

print(f"\nAgents Represented: {df['agent'].nunique()}/5 expected agents")
if df['agent'].nunique() == 5:
    print("SUCCESS: All 5 agents represented for comprehensive analysis!")

# Convert timestamps for temporal analysis
print("\nProcessing temporal data...")
df['created_at'] = pd.to_datetime(df['created_at'])
df['closed_at'] = pd.to_datetime(df['closed_at'])
df['merged_at'] = pd.to_datetime(df['merged_at'])

# User activity analysis
print("\nCOMPREHENSIVE USER ADOPTION ANALYSIS")
print("=" * 50)

# Basic user statistics
user_activity = df.groupby('user').agg({
    'id': 'count',
    'created_at': ['min', 'max'],
    'state': lambda x: (x == 'closed').sum(),
    'repo_url': 'nunique',
    'agent': lambda x: list(x.unique())[0]  # Get the agent (should be consistent)
}).round(3)

# Flatten column names
user_activity.columns = ['total_prs', 'first_pr', 'last_pr', 'closed_prs', 'unique_repos', 'agent']

# Calculate derived metrics
user_activity['success_rate'] = user_activity['closed_prs'] / user_activity['total_prs']
user_activity['days_active'] = (user_activity['last_pr'] - user_activity['first_pr']).dt.days
user_activity['repo_diversity'] = user_activity['unique_repos'] / user_activity['total_prs']

# User classification based on activity patterns
def classify_user_adoption(row):
    if row['total_prs'] == 1:
        return 'One-time User'
    elif row['total_prs'] <= 5:
        return 'Casual User'
    elif row['total_prs'] <= 20:
        return 'Regular User'
    else:
        return 'Power User'

user_activity['user_type'] = user_activity.apply(classify_user_adoption, axis=1)

# Calculate engagement metrics
def calculate_engagement_score(row):
    # Composite score based on activity, success, and diversity
    activity_score = min(row['total_prs'] / 20, 1)  # Normalize to max 20 PRs
    success_score = row['success_rate']
    diversity_score = min(row['repo_diversity'] * 5, 1)  # Normalize repo diversity
    return (activity_score + success_score + diversity_score) / 3

user_activity['engagement_score'] = user_activity.apply(calculate_engagement_score, axis=1)

# Display comprehensive results
print(f"USER ADOPTION METRICS:")
print(f"  Total unique users: {len(user_activity):,}")
print(f"  Average PRs per user: {user_activity['total_prs'].mean():.1f}")
print(f"  Median PRs per user: {user_activity['total_prs'].median():.1f}")
print(f"  Average success rate: {user_activity['success_rate'].mean():.1%}")
print(f"  Average engagement score: {user_activity['engagement_score'].mean():.2f}")

print(f"\nUSER TYPE DISTRIBUTION:")
user_type_stats = user_activity['user_type'].value_counts()
for user_type, count in user_type_stats.items():
    percentage = (count / len(user_activity)) * 100
    avg_prs = user_activity[user_activity['user_type'] == user_type]['total_prs'].mean()
    avg_success = user_activity[user_activity['user_type'] == user_type]['success_rate'].mean()
    print(f"  {user_type}: {count:,} users ({percentage:.1f}%) - Avg: {avg_prs:.1f} PRs, {avg_success:.1%} success")

print(f"\nTOP PERFORMING USERS:")
top_users = user_activity.nlargest(10, 'engagement_score')
for i, (user, stats) in enumerate(top_users.iterrows(), 1):
    print(f"  {i:2d}. {user}: {stats['total_prs']} PRs, {stats['success_rate']:.1%} success, {stats['unique_repos']} repos, score: {stats['engagement_score']:.2f}")

# Repository diversity analysis
print(f"\nREPOSITORY ENGAGEMENT:")
avg_repo_diversity = user_activity['repo_diversity'].mean()
multi_repo_users = (user_activity['unique_repos'] > 1).sum()
print(f"  Average repository diversity: {avg_repo_diversity:.2f}")
print(f"  Users contributing to multiple repos: {multi_repo_users:,} ({multi_repo_users/len(user_activity):.1%})")

# Temporal patterns
print(f"\nTEMPORAL ADOPTION PATTERNS:")
active_period = df['created_at'].max() - df['created_at'].min()
print(f"  Data spans: {active_period.days} days")
print(f"  Daily average PRs: {len(df) / active_period.days:.1f}")

# Long-term vs short-term users
long_term_users = (user_activity['days_active'] > 7).sum()
print(f"  Long-term users (>7 days active): {long_term_users:,} ({long_term_users/len(user_activity):.1%})")

# Agent-specific user patterns
agent_user_stats = df.groupby('agent')['user'].nunique()
print(f"\nAGENT USER ADOPTION:")
for agent, user_count in agent_user_stats.items():
    avg_prs_per_user = df[df['agent'] == agent].groupby('user')['id'].count().mean()
    print(f"  {agent}: {user_count:,} unique users, {avg_prs_per_user:.1f} avg PRs per user")

print(f"\nUSER ADOPTION ANALYSIS COMPLETE!")
print(f"Analysis based on: User activity patterns, success rates, repository diversity")
print(f"No external user data API required - comprehensive analysis from existing dataset!")

📈 RQ5: User Adoption Patterns Analysis
📅 Analysis Date: 2025-10-12 03:41:36

📂 Loading sample data...
Loading dataset from local file: ../data/raw/aidata.csv


Loaded sample of 10000 rows
✅ Loaded 10,000 PRs for user adoption analysis

👥 USER STATISTICS:
  Total Users: 1,699
  Average PRs per User: 5.9
  Median PRs per User: 1.0

📊 USER TYPE DISTRIBUTION:
  Newcomer: 1,643 users (96.7%)
  Regular: 55 users (3.2%)
  Experienced: 1 users (0.1%)

🔍 DateTime Analysis: ✅ Available
